In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, cross_validate

path = '/content/drive/MyDrive/Human_Vs_AI_Rewritten_Dataset_Bangla_Version.xlsx'
df = pd.read_excel(path)

print(df.shape)
print(df.columns.tolist())
df.head()

(4008, 7)
['ID', 'Pair_ID', 'Text', 'Source', 'Category', 'Word_Count', 'Label']


,ID,Pair_ID,Text,Source,Category,Word_Count,Label
0,bn_001,bn_001,"খাদ্যের উপাদান ছয়টি। সেগুলো হলো: শর্করা, আমিষ...",TextBook,Education,58,Human
1,bn_002,bn_001,"খাদ্যের ছয়টি প্রধান উপাদান হলো শর্করা, আমিষ, ...",ChatGPT,Education,50,AI
2,bn_003,bn_002,"কার্বন, হাইড্রোজেন, অক্সিজেন এবং নাইট্রোজেন - ...",TextBook,Education,54,Human
3,bn_004,bn_002,"কার্বন, হাইড্রোজেন, অক্সিজেন ও নাইট্রোজেন—এই চ...",ChatGPT,Education,53,AI
4,bn_005,bn_003,ফ্যাটি এসিড এবং গ্লিসারলের সমন্বয়ে স্নেহ পদার্...,TextBook,Education,61,Human


In [3]:
# Standardize source names
fix = {
    'chatgpt': 'ChatGPT',
    'claude': 'Claude',
    'deepseek': 'DeepSeek',
    'gemini': 'Gemini',
    'textbook': 'TextBook',
    'wikisource': 'WikiSource',
    'wikinews': 'WikiNews'
}

df['Source'] = df['Source'].astype(str).str.strip().str.lower().map(fix)

# Standardize labels
df['Label'] = df['Label'].astype(str).str.strip()

# Extract text length features directly from the text
df['word_count'] = df['Text'].astype(str).str.split().str.len()
df['char_count'] = df['Text'].astype(str).str.len()

# Encode the target variable: AI = 1, Human = 0
df['y'] = (df['Label'] == 'AI').astype(int)

# Display label distribution
print(df['Label'].value_counts())

# Display source distribution
print(df['Source'].value_counts())

# Check the correlation between the calculated and existing word counts
# The two values should be approximately equal
print(df[['word_count', 'Word_Count']].corr())

Label
Human    2004
AI       2004
Name: count, dtype: int64
Source
WikiSource    1204
DeepSeek       805
TextBook       550
ChatGPT        548
Gemini         343
Claude         308
WikiNews       250
Name: count, dtype: int64
            word_count  Word_Count
word_count         1.0         1.0
Word_Count         1.0         1.0


In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

X = df[['word_count', 'char_count']].values
y = df['y'].values
groups = df['Pair_ID'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

print("Train texts:", len(train_idx), "| Test texts:", len(test_idx))
print("Train pairs:", len(set(groups[train_idx])), "| Test pairs:", len(set(groups[test_idx])))
print("Pair overlap:", len(set(groups[train_idx]) & set(groups[test_idx])))  # 0 হওয়া উচিত

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(X[train_idx], y[train_idx])

pred  = clf.predict(X[test_idx])
proba = clf.predict_proba(X[test_idx])[:, 1]
y_test = y[test_idx]

results = pd.DataFrame({
    'Accuracy':  [accuracy_score(y_test, pred)],
    'Precision': [precision_score(y_test, pred)],
    'Recall':    [recall_score(y_test, pred)],
    'F1':        [f1_score(y_test, pred)],
    'ROC-AUC':   [roc_auc_score(y_test, proba)],
}).round(3)

print("Chance level: 0.500")
results

Train texts: 3206 | Test texts: 802
Train pairs: 1603 | Test pairs: 401
Pair overlap: 0
Chance level: 0.500


,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.571,0.579,0.519,0.547,0.598


In [ ]:
# Calculate the word-count ratio between AI-generated and human-written texts
pairs['ratio'] = pairs['word_count_a'] / pairs['word_count_h']

# Generate descriptive statistics for the word-count ratio
# including the 5th, 25th, 50th, 75th, and 95th percentiles
summary = pairs['ratio'].describe(
    percentiles=[.05, .25, .5, .75, .95]
).round(3)

print(summary)

# Calculate the percentage of paired texts whose word-count ratios
# fall within the specified tolerance intervals
for lo, hi in [(0.9, 1.1), (0.8, 1.2), (0.7, 1.3)]:
    share = pairs['ratio'].between(lo, hi).mean() * 100
    print(f"Ratio within [{lo}, {hi}]: {share:.1f}%")

# Test whether the word counts of AI-generated and human-written texts
# differ significantly using the paired Wilcoxon signed-rank test
w, p = stats.wilcoxon(
    pairs['word_count_a'],
    pairs['word_count_h']
)

print(f"\nWilcoxon signed-rank: W={w:.1f}, p={p:.2e}")

# Calculate the mean word-count difference between AI-generated
# and human-written texts
print(
    f"Mean diff (AI - Human) = "
    f"{(pairs['word_count_a'] - pairs['word_count_h']).mean():.2f} words"
)

count    2004.000
mean        1.103
std         0.186
min         0.360
5%          0.925
25%         1.000
50%         1.058
75%         1.160
95%         1.400
max         3.179
Name: ratio, dtype: float64
Ratio within [0.9, 1.1]: 59.5%
Ratio within [0.8, 1.2]: 80.6%
Ratio within [0.7, 1.3]: 89.6%

Wilcoxon signed-rank: W=263666.0, p=1.09e-141
Mean diff (AI - Human) = 4.80 words
